In [ ]:
import numpy as np
import scipy.signal

## Convolutional layer
def conv_forward(H_prev, V, b, stride=1, padding=0):
    (n, d_H_prev, d_L_prev, d_C_prev) = H_prev.shape
    (f, f, d_C_prev, d_C) = V.shape
    d_H = int((d_H_prev + 2 * padding - f) / stride) + 1
    d_L = int((d_L_prev + 2 * padding - f) / stride) + 1

    H_prev_pad = np.pad(H_prev, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode='constant', constant_values=(0, 0))
    Z = np.zeros((n, d_H, d_L, d_C))

    for i in range(n):
        h_prev_pad = H_prev_pad[i]
        for h in range(d_H):
            for l in range(d_L):
                for c in range(d_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = l * stride
                    horiz_end = horiz_start + f
                    h_slice = h_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    Z[i, h, l, c] = np.sum(h_slice * V[:, :, :, c]) + b[:, :, :, c]
    return Z

def conv_backward(dZ, H_prev, V, b, stride=1, padding=0):
    (n, d_H_prev, d_L_prev, d_C_prev) = H_prev.shape
    (f, f, d_C_prev, d_C) = V.shape
    (n, d_H, d_L, d_C) = dZ.shape

    H_prev_pad = np.pad(H_prev, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode='constant', constant_values=(0, 0))
    dH_prev_pad = np.zeros_like(H_prev_pad)
    dV = np.zeros_like(V)
    db = np.zeros_like(b)

    for i in range(n):
        h_prev_pad = H_prev_pad[i]
        dh_prev_pad = dH_prev_pad[i]
        for h in range(d_H):
            for l in range(d_L):
                for c in range(d_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = l * stride
                    horiz_end = horiz_start + f
                    h_slice = h_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    dh_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += V[:, :, :, c] * dZ[i, h, l, c]
                    dV[:, :, :, c] += h_slice * dZ[i, h, l, c]
                    db[:, :, :, c] += dZ[i, h, l, c]
        dH_prev_pad[i] = dh_prev_pad
    dH_prev = dH_prev_pad[:, padding:-padding, padding:-padding, :]
    return dH_prev, dV, db

## Pooling layer
def pool_forward(H_prev, f=2, stride=2, mode="max"):
    (n, d_H_prev, d_L_prev, d_C_prev) = H_prev.shape
    d_H = int(1 + (d_H_prev - f) / stride)
    d_L = int(1 + (d_L_prev - f) / stride)
    d_C = d_C_prev

    H = np.zeros((n, d_H, d_L, d_C))

    for i in range(n):
        for h in range(d_H):
            for l in range(d_L):
                for c in range(d_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = l * stride
                    horiz_end = horiz_start + f
                    h_slice = H_prev[i, vert_start:vert_end, horiz_start:horiz_end, c]

                    if mode == "max":
                        H[i, h, l, c] = np.max(h_slice)
                    elif mode == "average":
                        H[i, h, l, c] = np.mean(h_slice)
    return H

def pool_backward(dH, H_prev, f=2, stride=2, mode="max"):
    (n, d_H_prev, d_L_prev, d_C_prev) = H_prev.shape
    (n, d_H, d_L, d_C) = dH.shape

    dH_prev = np.zeros_like(H_prev)

    for i in range(n):
        h_prev = H_prev[i]
        for h in range(d_H):
            for l in range(d_L):
                for c in range(d_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = l * stride
                    horiz_end = horiz_start + f

                    if mode == "max":
                        h_slice = h_prev[vert_start:vert_end, horiz_start:horiz_end, c]
                        mask = h_slice == np.max(h_slice)
                        dH_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dH[i, h, l, c]
                    elif mode == "average":
                        da = dH[i, h, l, c]
                        shape = (f, f)
                        dH_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += distribute_value(da, shape)
    return dH_prev

def distribute_value(dz, shape):
    (d_H, d_L) = shape
    average = dz / (d_H * d_L)
    return np.ones(shape) * average

In [ ]:
## Fully connected layer
def fc_forward(H, V, b):
    Z = np.dot(H, V) + b
    return Z

def fc_backward(dZ, H, V, b):
    n = H.shape[0]
    dV = np.dot(H.T, dZ) / n
    db = np.sum(dZ, axis=0, keepdims=True) / n
    dH = np.dot(dZ, V.T)
    return dH, dV, db

In [30]:
class SimpleCNN:
    def __init__(self):
        self.parameters = self.initialize_parameters()

    def initialize_parameters(self):
        np.random.seed(1)
        parameters = {}
        parameters['V1'] = np.random.randn(3, 3, 1, 8) * 0.1
        parameters['b1'] = np.zeros((1, 1, 1, 8))
        parameters['V2'] = np.random.randn(3, 3, 8, 16) * 0.1
        parameters['b2'] = np.zeros((1, 1, 1, 16))
        
        # Compute the correct size for the flattened output from the convolutional layers
        conv_out_height = (64 - 2) // 2  # After first pooling layer
        conv_out_height = (conv_out_height - 2) // 2  # After second pooling layer
        conv_out_width = (64 - 2) // 2  # After first pooling layer
        conv_out_width = (conv_out_width - 2) // 2  # After second pooling layer
        flattened_size = conv_out_height * conv_out_width * 16
        
        parameters['V3'] = np.random.randn(flattened_size, 128) * 0.1
        parameters['b3'] = np.zeros((1, 128))
        parameters['V4'] = np.random.randn(128, 1) * 0.1
        parameters['b4'] = np.zeros((1, 1))
        return parameters

    def forward_propagation(self, X):
        parameters = self.parameters
        self.cache = {}

        # Conv Layer 1
        self.cache['Z1'] = conv_forward(X, parameters['V1'], parameters['b1'])
        self.cache['H1'] = np.maximum(0, self.cache['Z1'])  # ReLU Activation
        self.cache['P1'] = pool_forward(self.cache['H1'])

        # Conv Layer 2
        self.cache['Z2'] = conv_forward(self.cache['P1'], parameters['V2'], parameters['b2'])
        self.cache['H2'] = np.maximum(0, self.cache['Z2'])  # ReLU Activation
        self.cache['P2'] = pool_forward(self.cache['H2'])

        # Flatten
        self.cache['F'] = self.cache['P2'].reshape(self.cache['P2'].shape[0], -1)

        # Fully Connected Layer 1
        self.cache['Z3'] = fc_forward(self.cache['F'], parameters['V3'], parameters['b3'])
        self.cache['H3'] = np.maximum(0, self.cache['Z3'])  # ReLU Activation

        # Fully Connected Layer 2 (Output Layer)
        self.cache['Z4'] = fc_forward(self.cache['H3'], parameters['V4'], parameters['b4'])
        self.cache['H4'] = 1 / (1 + np.exp(-self.cache['Z4']))  # Sigmoid Activation

        return self.cache['H4']

    def backward_propagation(self, X, Y):
        parameters = self.parameters
        cache = self.cache

        n = X.shape[0]

        # Output layer
        dZ4 = cache['H4'] - Y
        dH3, dV4, db4 = fc_backward(dZ4, cache['H3'], parameters['V4'], parameters['b4'])

        # Fully connected layer 1
        dH3[cache['Z3'] <= 0] = 0  # ReLU backpropagation
        dH2_flat, dV3, db3 = fc_backward(dH3, cache['F'], parameters['V3'], parameters['b3'])

        # Reshape gradient to match P2 dimensions
        dH2 = dH2_flat.reshape(cache['P2'].shape)

        # Pooling layer 2
        dP2 = pool_backward(dH2, cache['H2'])
        dP2[cache['Z2'] <= 0] = 0  # ReLU backpropagation

        # Convolutional layer 2
        dH1, dV2, db2 = conv_backward(dP2, cache['P1'], parameters['V2'], parameters['b2'])

        # Pooling layer 1
        dP1 = pool_backward(dH1, cache['H1'])
        dP1[cache['Z1'] <= 0] = 0  # ReLU backpropagation

        # Convolutional layer 1
        dH_prev, dV1, db1 = conv_backward(dP1, X, parameters['V1'], parameters['b1'])

        # Update gradients
        grads = {'dV1': dV1, 'db1': db1, 'dV2': dV2, 'db2': db2, 'dV3': dV3, 'db3': db3, 'dV4': dV4, 'db4': db4}
        return grads

    def update_parameters(self, grads, learning_rate):
        for key in self.parameters.keys():
            self.parameters[key] -= learning_rate * grads['d' + key]
        return self.parameters

    def compute_cost(self, H4, Y):
        n = Y.shape[0]
        cost = -np.sum(Y * np.log(H4) + (1 - Y) * np.log(1 - H4)) / n
        cost = np.squeeze(cost)
        return cost

    def train(self, X_train, Y_train, X_test, Y_test, learning_rate=0.01, num_iterations=100):
        for i in range(num_iterations):
            Y_prediction_train = self.forward_propagation(X_train)
            cost = self.compute_cost(Y_prediction_train, Y_train)
            grads = self.backward_propagation(X_train, Y_train)
            self.update_parameters(grads, learning_rate)
            Y_prediction_test = self.forward_propagation(X_test)
            if i % 10 == 0:
                print(f"Cost after iteration {i}: {cost}")
                
        print("precision of training: {} %".format(100-np.mean(np.abs(Y_prediction_train-Y_train))*100))
        print("precision of test: {} %".format(100-np.mean(np.abs(Y_prediction_test-Y_test))*100))            

# Example of using this model

In [35]:
X_train = np.random.randn(10, 64, 64, 1)
Y_train = np.random.randint(0, 2, (10, 1))
X_test = np.random.randn(2, 64, 64, 1)
Y_test = np.random.randint(0, 2, (2, 1))

cnn = SimpleCNN()
cnn.train(X_train, Y_train, X_test, Y_test, learning_rate=0.01, num_iterations=100)

Cost after iteration 0: 1.5058016448160483
Cost after iteration 10: 0.23467889494950547
Cost after iteration 20: 0.13840500880357812
Cost after iteration 30: 0.09350147296891918
Cost after iteration 40: 0.06786276754139627
Cost after iteration 50: 0.05215866999704945
Cost after iteration 60: 0.04134051146250771
Cost after iteration 70: 0.03391860467239473
Cost after iteration 80: 0.02857417582211812
Cost after iteration 90: 0.02459437287208559
precision of training: 97.85151807157976 %
precision of test: 46.95456659724958 %
